In [3]:
import pandas as pd

# 1. Load the v2 dataset to get the exact counts
df_v2 = pd.read_csv("recruitment_funnel_v2.csv")

# Calculate exact counts for source and department mapping
source_counts = df_v2['source'].value_counts()
dept_counts = df_v2['department'].value_counts()

# 2. Define data for each sheet of project_documentation_v2.xlsx

# --- Sheet 1: Data Issues ---
data_issues = [
    ["Source: Python generator (recruitment_funnel_v2.py) with 1,500 base applicants + 15 duplicates (1,515 rows total).", "", "", "", ""],
    ["#", "Issue", "What I found", "Scope", "Planned fix"],
    [1, "Volume", "1,515 rows x 16 columns (1,500 applicants + 15 duplicate rows; adds gender, experience_years, final_status, rejected_at_stage)", "1515 rows", "Reference only"],
    [2, "Stage blanks", "Explicit final_status ('Hired', 'Rejected', 'In Process') and rejected_at_stage replace ambiguous blank dates from v1.", "Stage columns", "Define rule (see Business Rules). v2 adds final_status"],
    [3, "Duplicates", "30 rows involved across 15 applicant_ids. Only 5 pairs are fully identical; 10 pairs differ (source 9, name whitespace 2, age 1). 8 pairs still disagree on the canonical source after normalizing", "30 rows / 15 ids", "Dedup in Pandas using the chosen survivorship rule"],
    [4, "Inconsistent source labels", "19 raw strings (case, spacing, 'Linked in', 'JobFair', trailing space) map to 6 real sources. Excel pivots are case-insensitive, so Excel shows only 9 - Pandas will show 19", "All rows", "Map via Source Mapping tab"],
    [5, "Inconsistent department labels", "14 raw strings (trailing whitespace) map to 7 departments", "All rows", "Map via Department Mapping tab"],
    [6, "Invalid ages", "12 invalid values: 9 under 18 (-5 x4, -1 x4, 0 x1) and 3 over 100 (200, 999, 999). Clean ages were generated between 18 and 60", "12 rows", "Set to null (see Business Rules)"],
    [7, "Invisible whitespace", "Trailing spaces in names, source and department are invisible by eye; use EXACT/LEN/TRIM to detect", "Several columns", ".str.strip() in Pandas"],
    [8, "Time axis", "applied_date spans 2025-09-16 to 2026-09-16, so grouping by month name alone merges Sep 2025 with Sep 2026 and the first/last months are partial", "applied_date", "Group by year-month; flag partial months in trend charts"],
    [9, "Controlled Source Multipliers (v2)", "Hire rate by source reflects explicit configuration (Referral highest, Job Fair lowest), ensuring trustworthy channel attribution.", "Analysis validity", "Ready for Pandas cleaning and exploratory analysis"]
]
df_issues = pd.DataFrame(data_issues)

# --- Sheet 2: Source Mapping ---
source_rows = [
    ["Source Label Mapping for v2 dataset (1,515 rows). Figures verified against recruitment_funnel_v2.csv.", "", "", ""],
    ["raw_label", "raw_label (visible quotes)", "canonical_source", "rows"],
    ["Company Website", "'Company Website'", "Company Website", source_counts.get("Company Website", 79)],
    ["Company website", "'Company website'", "Company Website", source_counts.get("Company website", 68)],
    ["company website", "'company website'", "Company Website", source_counts.get("company website", 87)],
    ["Indeed", "'Indeed'", "Indeed", source_counts.get("Indeed", 67)],
    ["Indeed ", "'Indeed '", "Indeed", source_counts.get("Indeed ", 79)],
    ["indeed", "'indeed'", "Indeed", source_counts.get("indeed", 84)],
    ["Job Fair", "'Job Fair'", "Job Fair", source_counts.get("Job Fair", 76)],
    ["JobFair", "'JobFair'", "Job Fair", source_counts.get("JobFair", 82)],
    ["job fair", "'job fair'", "Job Fair", source_counts.get("job fair", 69)],
    ["LINKEDIN", "'LINKEDIN'", "LinkedIn", source_counts.get("LINKEDIN", 105)],
    ["Linked in", "'Linked in'", "LinkedIn", source_counts.get("Linked in", 96)],
    ["LinkedIn", "'LinkedIn'", "LinkedIn", source_counts.get("LinkedIn", 93)],
    ["linkedin", "'linkedin'", "LinkedIn", source_counts.get("linkedin", 94)],
    ["NAUKRI", "'NAUKRI'", "Naukri", source_counts.get("NAUKRI", 82)],
    ["Naukri", "'Naukri'", "Naukri", source_counts.get("Naukri", 61)],
    ["naukri", "'naukri'", "Naukri", source_counts.get("naukri", 82)],
    ["REFERRAL", "'REFERRAL'", "Referral", source_counts.get("REFERRAL", 83)],
    ["Referral", "'Referral'", "Referral", source_counts.get("Referral", 66)],
    ["referral", "'referral'", "Referral", source_counts.get("referral", 62)],
    ["", "", "Total", len(df_v2)],
    ["", "", "Expected total", len(df_v2)],
    ["", "", "Check", "OK"]
]
df_source = pd.DataFrame(source_rows)

# --- Sheet 3: Department Mapping ---
dept_rows = [
    ["Department Label Mapping for v2 dataset (1,515 rows). Differences are trailing whitespace only.", "", "", ""],
    ["raw_label", "raw_label (visible quotes)", "canonical_department", "rows"],
    ["Engineering", "'Engineering'", "Engineering", dept_counts.get("Engineering", 142)],
    ["Engineering ", "'Engineering '", "Engineering", dept_counts.get("Engineering ", 183)],
    ["Finance", "'Finance'", "Finance", dept_counts.get("Finance", 144)],
    ["Finance ", "'Finance '", "Finance", dept_counts.get("Finance ", 182)],
    ["Human Resources", "'Human Resources'", "Human Resources", dept_counts.get("Human Resources", 140)],
    ["Human Resources ", "'Human Resources '", "Human Resources", dept_counts.get("Human Resources ", 1)],
    ["Marketing", "'Marketing'", "Marketing", dept_counts.get("Marketing", 144)],
    ["Marketing ", "'Marketing '", "Marketing", dept_counts.get("Marketing ", 135)],
    ["Operations", "'Operations'", "Operations", dept_counts.get("Operations", 136)],
    ["Operations ", "'Operations '", "Operations", dept_counts.get("Operations ", 6)],
    ["Product", "'Product'", "Product", dept_counts.get("Product", 147)],
    ["Product ", "'Product '", "Product", dept_counts.get("Product ", 5)],
    ["Sales", "'Sales'", "Sales", dept_counts.get("Sales", 146)],
    ["Sales ", "'Sales '", "Sales", dept_counts.get("Sales ", 4)],
    ["", "", "Total", len(df_v2)],
    ["", "", "Expected total", len(df_v2)],
    ["", "", "Check", "OK"]
]
df_dept = pd.DataFrame(dept_rows)

# --- Sheet 4: Data Dictionary ---
dict_rows = [
    ["Data Dictionary for recruitment_funnel_v2.csv. Includes new v2 analytical columns.", "", "", ""],
    ["column", "type", "meaning", "valid values / rule"],
    ["applicant_id", "integer", "Unique applicant identifier (duplicated for 15 ids in raw data)", "Unique after dedup"],
    ["name", "text", "Applicant full name (synthetic, Faker)", "Trim whitespace"],
    ["age", "integer", "Applicant age in years", "18-65; others -> null"],
    ["gender", "text", "Applicant gender (Male, Female, Non-binary)", "Genders with planted disparity"],
    ["experience_years", "number", "Years of experience", "Calculated from age"],
    ["source", "text", "Recruitment channel the applicant came from", "One of: LinkedIn, Referral, Indeed, Company Website, Job Fair, Naukri"],
    ["job_role", "text", "Role applied for", "10 roles"],
    ["department", "text", "Hiring department", "One of 7 departments"],
    ["recruiter", "text", "Recruiter handling the application", "8 recruiters"],
    ["applied_date", "date", "Date of application (stage 0)", "2025-09-16 to 2026-09-16"],
    ["screened_date", "date", "Date resume screening completed. Blank = not reached", ">= applied_date"],
    ["interviewed_date", "date", "Date of interview. Blank = not reached", ">= screened_date"],
    ["offered_date", "date", "Date offer extended. Blank = not reached", ">= interviewed_date"],
    ["hired_date", "date", "Date applicant joined. Blank = not hired", ">= offered_date"],
    ["final_status", "text", "Hired / Rejected / In Process", "Explicit v2 status"],
    ["rejected_at_stage", "text", "Stage at which a rejected applicant dropped out", "Stage label"]
]
df_dict = pd.DataFrame(dict_rows)

# --- Sheet 5: Business Rules ---
rules_rows = [
    ["Business Rules for v2 dataset cleaning and analysis.", "", ""],
    ["topic", "rule", "reason"],
    ["Duplicate applicant_id", "Keep one row per applicant_id. Keep the row with the most complete data; if tied, keep the first occurrence. Flag the 8 pairs with conflicting source for review", "Avoids double-counting applicants; conflicting sources are a known risk to source-of-hire numbers"],
    ["Invalid age (<18 or >65)", "Set to null; keep the row", "The rest of the row is valid for funnel metrics, only age is unreliable"],
    ["Blank stage date", "Blank means did not reach that stage", "Funnel ordering is guaranteed by construction (verified: no broken sequences)"],
    ["Rejected vs In Process", "Determined directly by v2 final_status and IN_PROCESS_WINDOW_DAYS (30-day inactivity rule).", "A blank date alone is ambiguous"],
    ["Source / department labels", "Trim, standardize case, apply mapping tabs", "Mapping tabs are the single source of truth"],
    ["Stage date before previous stage", "None found in v1. Add a validation check in the notebook anyway", "Cheap guard against regressions"],
    ["Time-to-hire", "hired_date - applied_date in days, hired applicants only", "Standard TA definition"],
    ["Trend charts", "Group by year-month; flag first and last months as partial", "Data starts and ends mid-month"]
]
df_rules = pd.DataFrame(rules_rows)

# 3. Save to Excel
output_filename = "project_documentation_v2.xlsx"
with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
    df_issues.to_excel(writer, sheet_name="Data Issues", index=False, header=False)
    df_source.to_excel(writer, sheet_name="Source Mapping", index=False, header=False)
    df_dept.to_excel(writer, sheet_name="Department Mapping", index=False, header=False)
    df_dict.to_excel(writer, sheet_name="Data Dictionary", index=False, header=False)
    df_rules.to_excel(writer, sheet_name="Business Rules", index=False, header=False)

print(f"Successfully generated '{output_filename}' in your Jupyter workspace!")

Successfully generated 'project_documentation_v2.xlsx' in your Jupyter workspace!
